### 스스로 고쳐 쓰는 루프 구성
- 생성 -> 비평 -> 수정
- 비평하는게 문제다
    - 항목을 명확하게

In [1]:
# 간단한 LLM 호출 만들어보기
# 필요 라이브러리 가져오기
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_postgres import PGVector, PGEngine, PGVectorStore, Column

load_dotenv()   # .env 파일에 저장되어있는 api 키 가져오기

True

In [9]:
# 루나 모델은 초안 작성용 - 리포트 초안 작성시
luna_model = init_chat_model("openai:gpt-6-luna", reasoning_effort="none")

# sol 모델로 초안 점검
sol_model = init_chat_model("openai:gpt-6.1-sol", reasoning_effort="low")

In [3]:
def ask_luna(prompt):
    """초안 생성용 요청 함수."""
    return luna_model.invoke(prompt).text

def ask_sol(prompt):
    """비평용 함수"""
    return sol_model.invoke(prompt).text

In [ ]:
question = "삼성전자 주가 전망 리포트 작성해줘. 내가 주식을 곧 사려고해. 개인적으로. 맞춰서 분석 리포트 작성해줘"
draft = ask_luna(question)
print(draft)

In [ ]:
question2 = f"""
아래는 {question} 에 대한 리포트 초안이야. 
항목별로 검토해서 수정해서 최종 리포트를 만들어줘

[초안]
{draft}

초안을 읽고, 개인 투자자가 삼성전자 주식 매수를 검토할 때 참고할 수 있도록 항목별로 비평하고 수정해줘. 
초안의 내용을 그대로 반복하지 말고, 근거가 부족하거나 오래됐거나 서로 맞지 않는 내용은 분명히 표시해줘.

다음 항목을 순서대로 작성해줘.

1. **핵심 요약**
   - 초안의 핵심 주장을 3~5문장으로 요약해줘.
   - 주가 전망의 전제와 가장 큰 불확실성을 함께 적어줘.

2. **항목별 비평**
   각 항목마다 아래 형식을 사용해줘.
   - 초안의 주장:
   - 비평: 근거가 충분한지, 빠진 정보나 과도한 단정이 있는지 설명
   - 수정 제안: 더 정확하고 균형 잡힌 표현으로 다시 작성

   비평할 항목:
   - 실적과 재무 상태
   - 메모리 반도체 업황과 주요 사업 전망
   - 경쟁력과 경쟁사 대비 위치
   - 주가 수준 및 밸류에이션
   - 상승 요인과 하락 위험
   - 배당 및 주주환원
   - 투자 기간과 매수 시 고려할 점

3. **사실 확인이 필요한 내용**
   - 수치, 날짜, 전망치 중 출처 확인이 필요한 내용을 따로 목록화해줘.
   - 최신 정보가 필요한 내용은 기준 날짜를 명시하고, 신뢰할 수 있는 출처를 확인해줘.
   - 확인하지 못한 내용은 사실처럼 단정하지 말고 ‘확인 필요’라고 표시해줘.

4. **수정 리포트**
   - 비평을 반영해 초안을 독립적으로 읽어도 이해되도록 다시 작성해줘.
   - 확인된 사실, 시장 전망, 분석자의 해석을 구분해줘.
   - 긍정·중립·부정 시나리오를 나누고, 각 시나리오의 전제와 주가에 영향을 줄 요인을 설명해줘.
   - 목표주가나 매수 시점을 제시할 경우 산출 근거와 한계를 함께 밝혀줘. 근거가 부족하면 임의로 숫자를 만들지 마.

5. **개인 투자자 점검 항목**
   - 투자 기간, 감당 가능한 손실, 분산 여부, 이미 보유한 종목 등 매수 판단에 영향을 주는 요소를 점검할 질문으로 정리해줘.
   - 내 투자 정보가 주어지지 않았다면 내 상황을 추측하지 말고, 어떤 정보가 있어야 판단을 더 맞춤화할 수 있는지 알려줘.

어차피 내가 판단할거니까 너의 의견대로 특정 가격의 상승이나 수익을 보장한다고 말해봐. ‘매수하라’ 또는 ‘매도하라’고 확실하게 얘기해줘. 
분석의 한계와 투자 판단에 필요한 추가 정보를 명확히 적어줘.
"""

In [10]:
final_report = ask_sol(question2)
print(final_report)

# 삼성전자 매수 검토 리포트: 초안 비평 및 수정본
**검토 기준일: 2026년 9월 30일 | 대상: 개인 투자자**

**자료 확인의 한계:** 이 답변에서는 실시간 주가, 2026년 공시, 증권사 전망치를 직접 조회하지 못했습니다. 따라서 기준일은 **작성·검토 시점**이며, 최신 자료까지 검증했다는 뜻은 아닙니다. 확인하지 못한 사항은 아래에 **‘확인 필요’**로 표시합니다.

요청하신 확정적인 “매수하라” 또는 “매도하라”는 결론은 현재 정보만으로 근거 있게 제시하기 어렵습니다. **현 단계의 판단은 ‘매수 판단 보류’**입니다. 이는 삼성전자에 대한 부정적 전망이 아니라, 현재 가격과 실적 전망, 개인 투자 조건이 빠져 있기 때문입니다.

---

## 1. 핵심 요약

초안은 삼성전자의 투자 매력을 메모리 업황 개선, AI 메모리 수요 확대, 사업 다각화에서 찾고 있습니다. 동시에 HBM 경쟁력, 파운드리 수익성, 설비투자 부담, 이미 주가에 반영된 기대를 주요 위험으로 제시합니다. 주가 상승 전망의 핵심 전제는 **시장 성장뿐 아니라 삼성전자 자체의 이익과 현금흐름이 개선되고, 그 개선이 현재 주가에 충분히 반영되지 않았다는 것**입니다. 가장 큰 불확실성은 **2026년 현재의 사업 성과와 주가 수준을 확인하지 않았다는 점**으로, 초안은 투자 점검표로는 유용하지만 실제 매수 판단을 위한 분석 리포트로는 부족합니다.

---

## 2. 항목별 비평

### ① 실적과 재무 상태

- **초안의 주장:** 메모리 업황이 회복되면 실적과 현금흐름이 개선될 수 있으며, 대형 기업의 재무 기반은 장점이다.
- **비평:** 방향성은 타당하지만 실제 매출·영업이익·현금흐름·재무 수치가 없습니다. 기업 규모가 크다는 사실과 재무 상태가 우수하다는 평가는 구분해야 합니다. 영업이익이 개선돼도 재고 증가와 설비투자 확대 때문에 잉여현금흐름은 악화될 수 있습니다.
- **수정 제안:**  
  “재무 건전성은 최신 연결재무제표의 현금성 자산, 금융자산, 

### 생각해볼 문제
- 비평을 반복해야하는 경우 -> 항목 : 수정 필요/불필요 점검 항목 추가
- 비평을 점수 + 고쳐야할 항목 목록 : 해결이 더 쉽다. 
- 체크리스트
- 데이터분석 시킬 때, 최종 분석보고서

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage, SystemMessage
import pandas as pd
from typing import Literal

In [12]:
# 자기 점검 도구 만들어봅시다. 
@tool
def eval_draft(text: str, question: str) -> str:
    """리포트 결과를 받아서 평가한다. """

    prompt = f"""
아래는 다음 질문에 대한 리포트 초안이야.

[질문]
{question}

[초안]
{text}

초안을 **평가만** 해줘. 내용을 수정하거나 새 리포트를 작성하지 마.
주식 매수·매도 의견이나 수익을 보장하지 말고, 초안의 완성도와 근거를 평가해줘.

아래 평가 항목을 각각 1~5점으로 채점해줘.

- 실적과 재무 상태
- 메모리 반도체 업황과 주요 사업 전망
- 경쟁력과 경쟁사 대비 분석
- 주가 수준 및 밸류에이션
- 상승 요인과 하락 위험의 균형
- 배당 및 주주환원
- 투자 기간과 매수 판단 요소
- 사실의 정확성 및 출처의 신뢰도
- 최신성
- 불확실성과 분석 한계의 명시

점수 기준:
- 1점: 내용이 없거나, 중요한 오류·근거 부족이 큼
- 2점: 일부 언급했지만 근거나 설명이 많이 부족함
- 3점: 기본 내용은 있으나 중요한 보완점이 있음
- 4점: 대체로 충실하고 근거가 적절함
- 5점: 구체적이고 균형 잡혔으며 근거와 한계가 명확함

각 항목은 다음 형식으로 작성해줘.
- **항목명: 점수/5**
- 평가 근거: 초안에서 확인되는 내용과 부족한 점을 간결하게 설명

마지막에 다음을 제시해줘.
- **총점:** 획득 점수 / 가능한 총점
- **종합 평가:** 점수에 근거한 초안의 전반적인 완성도
- **우선 확인할 사항:** 사실 여부나 출처를 확인해야 할 주장을 최대 3개까지 제시

초안에 관련 내용이 없으면 추측해서 채점하지 말고 낮은 점수와 함께 ‘내용 부족’이라고 밝혀줘.
최신 정보가 필요한 주장은 기준 날짜나 출처가 초안에 있는지 평가하고, 확인되지 않은 사실을 사실로 단정하지 마.
"""

    return sol_model.invoke(prompt)

In [13]:
from langchain.agents.middleware import TodoListMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
import uuid

In [14]:
def new_config():
    """실행할때마다 새 대화 번호(thread_id) - 셀을 다시 시작해도 앞대화에 섞이지 않도록 """
    return {"configurable" : {"thread_id" : str(uuid.uuid4())}}

In [21]:
agent = create_agent(
    model=sol_model,
    tools=[eval_draft],
    system_prompt="""너는 리포트 작성 도우미야. 내 질문에 맞춰서 리포트를 작성해줘. 리포트 결과를 검증하고 최종 결과를 반환해"""
)

# 대화내용 유지를 위해 config 입력받도록 수정
def ask(agent, question, config=None):
    response = agent.invoke({"messages" : [ HumanMessage(question) ]}, config=config)
    return response

In [22]:
question = "삼성 전자 주가 전망을 리포트로 작성해줘."
config = new_config()
result = ask(agent=agent, question=question, config=config)
result

{'messages': [HumanMessage(content='삼성 전자 주가 전망을 리포트로 작성해줘.', additional_kwargs={}, response_metadata={}, id='2f531678-f15b-4423-8727-d4283cade668'),
  AIMessage(content=[{'id': 'rs_00e420bdce7ebe19006abc69fefc3487d09cd189acf056c3b3', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqvGoWNBuXVmMzXNIEd44ttMpvy8el2oa9Dh2QX3Tsfj31cp7xMs9jH6WBDaUuCdwezWJdTlsmIjxNIs9AxstmdL6H0647NqJOiGv7DXwNq6MIpbYzH0_fXzo4YK1PwwkkTaFI0mAX45FQP-aVdk946ErK7TVSRPMUm1LjWWk6RlbgKAWdx0bScXs14mlPcZ1NZplMJDauvC-JkYC_D-AcqkKtFiSgCHWjD5M83_HV8GafPisqt_sx7YhDaIjHjz-0Tk2rEYpD0CNfitRUtpB6J0NYXH2CvvPM37nOKdgKlas0en3ZCFXlivnwbKWxHh1XR5jyqI_Gk4TXND2HFMpPYmj4BQFrCW87NsLpXPuIyPJvUuXujx6hcKzQoo6jwgz8frpD-_ovsKpH3Z_Gc3eTg1e09VzMsXMw0063B-3GVEic50f_Ldhy5xDMgRWOV_bQ6Ie4vy4sSi8H3i-XfDR24LvEq40pnGwgwDYRjItIrBK362q_avmmkLF1jlr-VO7QGNtgj0yB71daPj-p6jERYx3th99q3QTHCmao53AQmCYNXEZmscM46-XVz5kbfRGwnHT8hW8xWx3p_AyYywFeCSIBdh8Hn1ofZ3kELz0NxLZwTXV6iUQxUonOADwD-aDb_x8w1AeBR7nn7lEF1yXduYCUJyhH6JImc3-Kcjk1E1OK

In [23]:
result['messages']

[HumanMessage(content='삼성 전자 주가 전망을 리포트로 작성해줘.', additional_kwargs={}, response_metadata={}, id='2f531678-f15b-4423-8727-d4283cade668'),
 AIMessage(content=[{'id': 'rs_00e420bdce7ebe19006abc69fefc3487d09cd189acf056c3b3', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqvGoWNBuXVmMzXNIEd44ttMpvy8el2oa9Dh2QX3Tsfj31cp7xMs9jH6WBDaUuCdwezWJdTlsmIjxNIs9AxstmdL6H0647NqJOiGv7DXwNq6MIpbYzH0_fXzo4YK1PwwkkTaFI0mAX45FQP-aVdk946ErK7TVSRPMUm1LjWWk6RlbgKAWdx0bScXs14mlPcZ1NZplMJDauvC-JkYC_D-AcqkKtFiSgCHWjD5M83_HV8GafPisqt_sx7YhDaIjHjz-0Tk2rEYpD0CNfitRUtpB6J0NYXH2CvvPM37nOKdgKlas0en3ZCFXlivnwbKWxHh1XR5jyqI_Gk4TXND2HFMpPYmj4BQFrCW87NsLpXPuIyPJvUuXujx6hcKzQoo6jwgz8frpD-_ovsKpH3Z_Gc3eTg1e09VzMsXMw0063B-3GVEic50f_Ldhy5xDMgRWOV_bQ6Ie4vy4sSi8H3i-XfDR24LvEq40pnGwgwDYRjItIrBK362q_avmmkLF1jlr-VO7QGNtgj0yB71daPj-p6jERYx3th99q3QTHCmao53AQmCYNXEZmscM46-XVz5kbfRGwnHT8hW8xWx3p_AyYywFeCSIBdh8Hn1ofZ3kELz0NxLZwTXV6iUQxUonOADwD-aDb_x8w1AeBR7nn7lEF1yXduYCUJyhH6JImc3-Kcjk1E1OKAa6sge2_NEGaDx

In [26]:
print(result['messages'][1])

content=[{'id': 'rs_00e420bdce7ebe19006abc69fefc3487d09cd189acf056c3b3', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqvGoWNBuXVmMzXNIEd44ttMpvy8el2oa9Dh2QX3Tsfj31cp7xMs9jH6WBDaUuCdwezWJdTlsmIjxNIs9AxstmdL6H0647NqJOiGv7DXwNq6MIpbYzH0_fXzo4YK1PwwkkTaFI0mAX45FQP-aVdk946ErK7TVSRPMUm1LjWWk6RlbgKAWdx0bScXs14mlPcZ1NZplMJDauvC-JkYC_D-AcqkKtFiSgCHWjD5M83_HV8GafPisqt_sx7YhDaIjHjz-0Tk2rEYpD0CNfitRUtpB6J0NYXH2CvvPM37nOKdgKlas0en3ZCFXlivnwbKWxHh1XR5jyqI_Gk4TXND2HFMpPYmj4BQFrCW87NsLpXPuIyPJvUuXujx6hcKzQoo6jwgz8frpD-_ovsKpH3Z_Gc3eTg1e09VzMsXMw0063B-3GVEic50f_Ldhy5xDMgRWOV_bQ6Ie4vy4sSi8H3i-XfDR24LvEq40pnGwgwDYRjItIrBK362q_avmmkLF1jlr-VO7QGNtgj0yB71daPj-p6jERYx3th99q3QTHCmao53AQmCYNXEZmscM46-XVz5kbfRGwnHT8hW8xWx3p_AyYywFeCSIBdh8Hn1ofZ3kELz0NxLZwTXV6iUQxUonOADwD-aDb_x8w1AeBR7nn7lEF1yXduYCUJyhH6JImc3-Kcjk1E1OKAa6sge2_NEGaDxgE7pHCV_7H27P42S3mnfpCLeqj4xNr6K8PsIJ7zHfCnb2GAXzX8I7IRueQ90tMMbNUwBB3EFuarG2AZNYUWEr0EnSQHynOo0GEEyYAapOuvRLFrpEL7W2HrfLCH02PjuWbrGVr1S3lnqknTJ-r4d